# Lab 2: Neural Network Tuning and Model Comparison

In the previous lab, we trained a basic feedforward neural network and observed that lower training loss does not always imply better generalization. In this lab, we use the validation set more systematically to compare neural network configurations and evaluate model performance.

We will compare neural networks with different hidden layer sizes and different levels of $L_2$ regularization. We will also compare the neural network to logistic regression on the same classification task.

## 1. Setup

We begin by importing the packages used in this lab and setting a random seed.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 2. Recreate the dataset and training pipeline

We rebuild the dataset split, preprocessing steps, and neural network training function as in the previous lab.



In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    stratify=y_train_full,
    random_state=SEED
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.to_numpy().reshape(-1, 1), dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.to_numpy().reshape(-1, 1), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy().reshape(-1, 1), dtype=torch.float32)

batch_size = 32


In [ ]:
class BasicNN(nn.Module):
    def __init__(self, input_dim, hidden_units):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, 1)
        )

    def forward(self, x):
        return self.network(x)


def train_model(
    hidden_units=16,
    lr=0.01,
    epochs=200,
    weight_decay=0.0,
    seed=SEED
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model = BasicNN(input_dim=X_train_tensor.shape[1], hidden_units=hidden_units)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator
    )

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": []
    }

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []

        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_tensor)
            val_loss = loss_fn(val_logits, y_val_tensor).item()

        history["epoch"].append(epoch)
        history["train_loss"].append(np.mean(batch_losses))
        history["val_loss"].append(val_loss)

    history = pd.DataFrame(history)
    return model, history


def predict_prob(model, X_tensor):
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        probs = torch.sigmoid(logits)
    return probs.numpy().ravel()


## 3. Compare different hidden layer sizes

We begin by comparing several hidden layer sizes while keeping the rest of the training setup fixed.

This allows us to examine how model flexibility affects validation and test performance.

In [ ]:
hidden_unit_grid = [4, 8, 16, 32, 64]

hidden_results = []
hidden_histories = {}

for hidden_units in hidden_unit_grid:
    model_tmp, history_tmp = train_model(
        hidden_units=hidden_units,
        lr=0.01,
        epochs=200,
        weight_decay=0.0
    )

    val_prob_tmp = predict_prob(model_tmp, X_val_tensor)
    test_prob_tmp = predict_prob(model_tmp, X_test_tensor)

    hidden_results.append({
        "hidden_units": hidden_units,
        "best_epoch": int(history_tmp.loc[history_tmp["val_loss"].idxmin(), "epoch"]),
        "best_val_loss": history_tmp["val_loss"].min(),
        "final_train_loss": history_tmp["train_loss"].iloc[-1],
        "final_val_loss": history_tmp["val_loss"].iloc[-1],
        "validation_accuracy": accuracy_score(y_val, (val_prob_tmp >= 0.5).astype(int)),
        "validation_auc": roc_auc_score(y_val, val_prob_tmp),
        "test_accuracy": accuracy_score(y_test, (test_prob_tmp >= 0.5).astype(int)),
        "test_auc": roc_auc_score(y_test, test_prob_tmp)
    })

    hidden_histories[hidden_units] = history_tmp

hidden_results = pd.DataFrame(hidden_results)

plt.figure(figsize=(7, 4))
for hidden_units in hidden_unit_grid:
    plt.plot(
        hidden_histories[hidden_units]["epoch"],
        hidden_histories[hidden_units]["val_loss"],
        label=f"{hidden_units} units"
    )

plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Validation loss across hidden layer sizes")
plt.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=len(hidden_unit_grid),
    frameon=False
)
plt.tight_layout()
plt.show()

## Interpretation of the hidden-layer comparison

The validation curves show that increasing the number of hidden units does not necessarily improve generalization. In this example, the smaller and medium-sized networks achieve lower validation loss and remain more stable over training, whereas the larger networks begin to overfit earlier and more severely.

This pattern illustrates a central point in model tuning: a more flexible neural network can fit the training data more aggressively, but that added flexibility may worsen out-of-sample performance.


In [ ]:
hidden_results = hidden_results.sort_values(
    ["best_val_loss", "validation_auc"],
    ascending=[True, False]
)

hidden_results.round(4)

## 4. Compare different values of weight decay

We now fix the hidden layer size and compare several values of weight decay.

Weight decay adds an $L_2$ penalty during optimization. This can improve generalization by discouraging excessively large weights and reducing overfitting.

In [ ]:
best_hidden_row = hidden_results.iloc[0].copy()
best_hidden_units = int(best_hidden_row["hidden_units"])
print("Selected hidden units from the previous section:", best_hidden_units)

weight_decay_grid = [0.0, 0.0001, 0.001, 0.01, 0.1]

wd_results = []
wd_histories = {}

for wd in weight_decay_grid:
    model_tmp, history_tmp = train_model(
        hidden_units=best_hidden_units,
        lr=0.01,
        epochs=200,
        weight_decay=wd
    )

    val_prob_tmp = predict_prob(model_tmp, X_val_tensor)
    test_prob_tmp = predict_prob(model_tmp, X_test_tensor)

    wd_results.append({
        "weight_decay": wd,
        "best_epoch": int(history_tmp.loc[history_tmp["val_loss"].idxmin(), "epoch"]),
        "best_val_loss": history_tmp["val_loss"].min(),
        "final_train_loss": history_tmp["train_loss"].iloc[-1],
        "final_val_loss": history_tmp["val_loss"].iloc[-1],
        "validation_accuracy": accuracy_score(y_val, (val_prob_tmp >= 0.5).astype(int)),
        "validation_auc": roc_auc_score(y_val, val_prob_tmp),
        "test_accuracy": accuracy_score(y_test, (test_prob_tmp >= 0.5).astype(int)),
        "test_auc": roc_auc_score(y_test, test_prob_tmp)
    })

    wd_histories[wd] = history_tmp

wd_results = pd.DataFrame(wd_results)

plt.figure(figsize=(7, 4))
for wd in weight_decay_grid:
    plt.plot(
        wd_histories[wd]["epoch"],
        wd_histories[wd]["val_loss"],
        label=f"wd={wd}"
    )

plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Validation loss across weight decay values")
plt.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=len(weight_decay_grid),
    frameon=False
)
plt.tight_layout()
plt.show()

## Interpretation of the regularization comparison

The validation curves show how regularization changes the training dynamics of the neural network. Smaller values of weight decay allow the model to fit more freely, which can lead to stronger overfitting. Larger values of weight decay constrain the model more strongly, which may improve stability but can also reduce flexibility if the penalty becomes too large.



In [ ]:
wd_results = wd_results.sort_values(
    ["best_val_loss", "validation_auc"],
    ascending=[True, False]
)

wd_results.round(4)


In [ ]:
best_wd_row = wd_results.iloc[0].copy()
best_wd_row


## 5. Fit the selected neural network

We now combine the preferred hidden layer size and the preferred weight decay value, then fit the selected neural network again and evaluate its performance.


In [ ]:
best_hidden_units = int(best_hidden_row["hidden_units"])
best_weight_decay = float(best_wd_row["weight_decay"])

print("Chosen hidden units:", best_hidden_units)
print("Chosen weight decay:", best_weight_decay)


In [ ]:
best_model, best_history = train_model(
    hidden_units=best_hidden_units,
    lr=0.01,
    epochs=200,
    weight_decay=best_weight_decay
)

best_val_prob = predict_prob(best_model, X_val_tensor)
best_test_prob = predict_prob(best_model, X_test_tensor)

best_val_pred = (best_val_prob >= 0.5).astype(int)
best_test_pred = (best_test_prob >= 0.5).astype(int)


In [ ]:
best_results = pd.DataFrame({
    "model": ["selected neural network"],
    "hidden_units": [best_hidden_units],
    "weight_decay": [best_weight_decay],
    "validation_accuracy": [accuracy_score(y_val, best_val_pred)],
    "validation_auc": [roc_auc_score(y_val, best_val_prob)],
    "test_accuracy": [accuracy_score(y_test, best_test_pred)],
    "test_auc": [roc_auc_score(y_test, best_test_prob)]
})

best_results.round(4)


## 6. Final comparison

We now compare the selected neural network with the earlier neural network that used 16 hidden units and no weight decay.


In [ ]:
baseline_model, baseline_history = train_model(
    hidden_units=16,
    lr=0.01,
    epochs=200,
    weight_decay=0.0
)

baseline_test_prob = predict_prob(baseline_model, X_test_tensor)
baseline_test_pred = (baseline_test_prob >= 0.5).astype(int)


In [ ]:
final_comparison = pd.DataFrame({
    "model": [
        "baseline neural network",
        "selected neural network"
    ],
    "hidden_units": [
        16,
        best_hidden_units
    ],
    "weight_decay": [
        0.0,
        best_weight_decay
    ],
    "test_accuracy": [
        accuracy_score(y_test, baseline_test_pred),
        accuracy_score(y_test, best_test_pred)
    ],
    "test_auc": [
        roc_auc_score(y_test, baseline_test_prob),
        roc_auc_score(y_test, best_test_prob)
    ]
})

final_comparison.sort_values("test_auc", ascending=False).round(4)


## 7. Summary

In this lab, we used the validation set to compare neural network configurations more systematically.

We first compared different hidden layer sizes and found that increasing model size did not automatically improve validation performance. We then fixed the hidden layer size and compared several values of weight decay, showing how regularization can influence overfitting and generalization.

After selecting a preferred configuration, we refit the model and compared it with the earlier baseline neural network. This workflow illustrates an important principle: model tuning should be guided by out-of-sample performance rather than training loss alone.


## 8. Additional practice

Try one or more of the following:

1. Add 128 hidden units to the hidden-unit grid. Does validation performance continue to improve?
2. Add smaller values between $0$ and $0.001$ to the weight decay grid. Do the results change meaningfully?
3. Change the learning rate from $0.01$ to $0.001$ and compare the validation-loss curves.
4. Compare the best epoch across different values of weight decay. Do stronger penalties change when the minimum validation loss occurs?
5. Write two or three sentences explaining why the validation set is useful when selecting neural network hyperparameters.
